In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import joblib

# Carregamento dos Dados

In [4]:
df = pd.read_csv("data\processed\dataset.csv")

In [5]:
df.head(10)

,Unnamed: 0,id_registro,texto_limpo,canal_origem,data,classe_macro,classe_detalhada,tamanho_texto,qntd_palavras_texto
0,0,1,erro recorrente sistema tentar acessar determi...,sistema,2025-03-08,Problemas Técnicos,Erro de Sistema / Aplicação,67,7
1,1,2,pedido integracao sincronizacao dados,email,2025-03-22,Solicitações Operacionais,Integração com Sistemas Externos,37,4
2,2,3,aguardando posicionamento,formulario,2025-03-02,Outros,Mensagem Genérica,25,2
3,3,4,sistema apresentou erro inesperado executar fu...,chat,2025-03-17,Problemas Técnicos,Erro de Sistema / Aplicação,69,7
4,5,6,registro automatico sistema,formulario,2025-03-19,Outros,Registro Automático,27,3
5,6,7,efetuei pagamento consta pendente sistema,formulario,2025-03-11,Financeiro e Cobrança,Pagamento Realizado mas Não Compensado,41,5
6,7,8,identifiquei cobranca duplicada fatura,sistema,2025-03-04,Financeiro e Cobrança,Cobrança Indevida / Duplicada,38,4
7,8,9,nao possivel exportar relatorio solicitado,sistema,2025-03-28,Problemas Técnicos,Erro de Exportação / Relatórios,42,5
8,9,10,preciso reenviar boleto pagamento,email,2025-03-24,Financeiro e Cobrança,Segunda Via de Boleto,33,4
9,10,11,atendimento excelente,email,2025-03-18,Feedback e Experiência,Feedback Positivo,21,2


# Divisão dos Dados

In [6]:
# usando stratify por classe macro, pois a classe detalhada tem poucos exemplos por classe e não cobre totalmente o conjunto de teste dessa forma, quebrando o código

X = df["texto_limpo"]
y_macro = df["classe_macro"]
y_detail = df['classe_detalhada']

X_train, X_test, y_macro_train, y_macro_test, y_detail_train, y_detail_test = train_test_split(
    X,
    y_macro,
    y_detail,
    test_size=0.2,
    random_state=42,
    stratify=y_macro
)

print("Treino:", len(X_train))
print("Teste:", len(X_test))

Treino: 62
Teste: 16


In [7]:
# criando dfs de treino e teste para facilitar treino e avaliação

train_df = pd.DataFrame({
    "texto_limpo": X_train.values,
    "classe_macro": y_macro_train.values,
    "classe_detalhada": y_detail_train.values
}).reset_index(drop=True)

test_df = pd.DataFrame({
    "texto_limpo": X_test.values,
    "classe_macro": y_macro_test.values,
    "classe_detalhada": y_detail_test.values
}).reset_index(drop=True)

train_df.head()

,texto_limpo,classe_macro,classe_detalhada
0,pedido agendamento treinamento,Solicitações Operacionais,Solicitação de Treinamento
1,dificuldade navegacao sistema,Feedback e Experiência,Crítica de Usabilidade
2,erro recorrente sistema tentar acessar determi...,Problemas Técnicos,Erro de Sistema / Aplicação
3,mensagem detalhes especificos,Outros,Mensagem Genérica
4,aplicacao ar desde inicio manha,Problemas Técnicos,Indisponibilidade / Queda do Serviço


# Modelos

In [8]:
# utilizando lr e svm pela simplicidade e bom desempenho em geral em dados mais simples

def criar_modelos():

    return {
        "logreg": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )),
            ("clf", LogisticRegression(
                max_iter=3000,
                class_weight="balanced"
            ))
        ]),
        "linear_svc": Pipeline([
            ("tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )),
            ("clf", LinearSVC(
                class_weight="balanced"
            ))
        ])
    }

In [9]:
# função para visualizar métricas importantes do desempenho dos modelos como f1-score e acurácia

def calcular_metricas(y_true, y_pred):
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

In [10]:
# criando modelos, treinando no conjunto de treino e motando dataframe com os resultados do conjunto de teste para comparação

resultados_macro = []
modelos_macro_candidatos = criar_modelos()

for nome_modelo, modelo in modelos_macro_candidatos.items():
    modelo.fit(train_df["texto_limpo"], train_df["classe_macro"])
    preds = modelo.predict(test_df["texto_limpo"])
    
    metricas = calcular_metricas(test_df["classe_macro"], preds)
    metricas["modelo"] = nome_modelo
    resultados_macro.append(metricas)

df_resultados_macro = pd.DataFrame(resultados_macro).sort_values(
    by=["f1_macro", "accuracy"],
    ascending=False
).reset_index(drop=True)

df_resultados_macro

,accuracy,f1_macro,f1_weighted,modelo
0,0.875,0.880635,0.875794,logreg
1,0.875,0.880635,0.875794,linear_svc


Modelos tiveram desempenho semelhante, então será escolhido um arbitrariamente (logreg)

In [11]:
modelo_macro = modelos_macro_candidatos["logreg"]

# Classificação Hieráquica

In [12]:
# adotando regressão logística para mantér o padrão e também reportar confiança com mais qualidade

def criar_especialista():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            max_df=1.0,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced"
        ))
    ])

In [13]:
modelos_especialistas = {}
resumo_especialistas = []

macros = sorted(train_df["classe_macro"].unique())

for macro in macros:
    train_macro = train_df[train_df["classe_macro"] == macro].copy()
    test_macro = test_df[test_df["classe_macro"] == macro].copy()

    n_amostras_treino = len(train_macro)
    n_amostras_teste = len(test_macro)
    n_classes_detalhadas = train_macro["classe_detalhada"].nunique()
    classes_detalhadas = sorted(train_macro["classe_detalhada"].unique().tolist())

    # se só existe uma classe detalhada, não precisa treinar especialista
    if n_classes_detalhadas < 2:
        resumo_especialistas.append({
            "classe_macro": macro,
            "n_amostras_treino": n_amostras_treino,
            "n_amostras_teste": n_amostras_teste,
            "n_classes_detalhadas": n_classes_detalhadas,
            "especialista_treinado": False,
            "motivo": "apenas_uma_classe_detalhada",
            "classes_detalhadas": classes_detalhadas
        })
        continue

    modelo_especialista = criar_especialista()

    try:
        modelo_especialista.fit(
            train_macro["texto_limpo"],
            train_macro["classe_detalhada"]
        )

        modelos_especialistas[macro] = modelo_especialista

        resumo_especialistas.append({
            "classe_macro": macro,
            "n_amostras_treino": n_amostras_treino,
            "n_amostras_teste": n_amostras_teste,
            "n_classes_detalhadas": n_classes_detalhadas,
            "especialista_treinado": True,
            "motivo": None,
            "classes_detalhadas": classes_detalhadas
        })

    except Exception as e:
        resumo_especialistas.append({
            "classe_macro": macro,
            "n_amostras_treino": n_amostras_treino,
            "n_amostras_teste": n_amostras_teste,
            "n_classes_detalhadas": n_classes_detalhadas,
            "especialista_treinado": False,
            "motivo": f"erro_no_treino: {str(e)}",
            "classes_detalhadas": classes_detalhadas
        })

In [14]:
df_resumo_especialistas = pd.DataFrame(resumo_especialistas)
df_resumo_especialistas

,classe_macro,n_amostras_treino,n_amostras_teste,n_classes_detalhadas,especialista_treinado,motivo,classes_detalhadas
0,Feedback e Experiência,10,2,4,True,None,"[Crítica de Usabilidade, Feedback Positivo, Su..."
1,Financeiro e Cobrança,14,4,6,True,None,"[Cobrança Indevida / Duplicada, Dúvidas sobre ..."
2,Outros,7,2,3,True,None,"[Agradecimento Simples, Mensagem Genérica, Reg..."
3,Problemas Técnicos,14,4,6,True,None,"[Erro de Autenticação e Acesso, Erro de Export..."
4,Solicitações Operacionais,17,4,7,True,None,"[Alteração de Plano / Contrato, Atualização Ca..."


In [15]:
def prever_hierarquico(textos, modelo_macro, modelos_especialistas):
    macro_preds = modelo_macro.predict(textos)

    detail_preds = []
    especialista_usado = []

    for texto, macro_pred in zip(textos, macro_preds):
        modelo_especialista = modelos_especialistas.get(macro_pred)
        detail_pred = modelo_especialista.predict([texto])[0]
        especialista_usado.append(True)

        detail_preds.append(detail_pred)

    return (
        np.array(macro_preds),
        np.array(detail_preds),
        np.array(especialista_usado)
    )

In [16]:
macro_pred_test, detail_pred_test, especialista_usado_test = prever_hierarquico(
    textos=test_df["texto_limpo"].values,
    modelo_macro=modelo_macro,
    modelos_especialistas=modelos_especialistas
)

In [17]:
predicoes_test = test_df.copy()

predicoes_test["macro_pred"] = macro_pred_test
predicoes_test["detail_pred"] = detail_pred_test
predicoes_test["especialista_usado"] = especialista_usado_test

predicoes_test["macro_correta"] = predicoes_test["classe_macro"] == predicoes_test["macro_pred"]
predicoes_test["detail_correta"] = predicoes_test["classe_detalhada"] == predicoes_test["detail_pred"]

predicoes_test

,texto_limpo,classe_macro,classe_detalhada,macro_pred,detail_pred,especialista_usado,macro_correta,detail_correta
0,pedido alteracao cadastral,Solicitações Operacionais,Atualização Cadastral,Solicitações Operacionais,Alteração de Plano / Contrato,True,True,False
1,solicito reprocessamento dados incorretos,Solicitações Operacionais,Reprocessamento / Correção de Dados,Solicitações Operacionais,Atualização Cadastral,True,True,False
2,solicito criacao relatorio personalizado,Solicitações Operacionais,Customização de Relatórios,Solicitações Operacionais,Customização de Relatórios,True,True,True
3,pagamento realizado porem nao identificado,Financeiro e Cobrança,Pagamento Realizado mas Não Compensado,Financeiro e Cobrança,Pagamento Realizado mas Não Compensado,True,True,True
4,duvidas sobre valor fatura,Financeiro e Cobrança,Dúvidas sobre Fatura / Valores,Financeiro e Cobrança,Dúvidas sobre Fatura / Valores,True,True,True
5,sistema poderia rapido,Feedback e Experiência,Sugestão de Melhoria,Feedback e Experiência,Crítica de Usabilidade,True,True,False
6,obrigado ajuda,Outros,Agradecimento Simples,Outros,Mensagem Genérica,True,True,False
7,aplicacao falhou durante uso normal exibindo m...,Problemas Técnicos,Erro de Sistema / Aplicação,Problemas Técnicos,Indisponibilidade / Queda do Serviço,True,True,False
8,sistema apresentou erro inesperado executar fu...,Problemas Técnicos,Erro de Sistema / Aplicação,Problemas Técnicos,Erro de Autenticação e Acesso,True,True,False
9,ha lentidao excessiva navegar telas,Problemas Técnicos,Problemas de Performance / Lentidão,Outros,Mensagem Genérica,True,False,False


# Avaliação Final

In [18]:
def calcular_metricas(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

In [19]:
metricas_macro = calcular_metricas(test_df["classe_macro"], macro_pred_test)
metricas_macro

{'accuracy': 0.875,
 'f1_macro': 0.8806349206349207,
 'f1_weighted': 0.8757936507936508}

In [20]:
metricas_detail = calcular_metricas(test_df["classe_detalhada"], detail_pred_test)
metricas_detail

{'accuracy': 0.4375,
 'f1_macro': 0.31746031746031744,
 'f1_weighted': 0.41666666666666663}

In [21]:
print("=== REPORT MACRO ===")
print(classification_report(test_df["classe_macro"], macro_pred_test, zero_division=0))

=== REPORT MACRO ===
                           precision    recall  f1-score   support

   Feedback e Experiência       1.00      1.00      1.00         2
    Financeiro e Cobrança       1.00      0.75      0.86         4
                   Outros       0.67      1.00      0.80         2
       Problemas Técnicos       1.00      0.75      0.86         4
Solicitações Operacionais       0.80      1.00      0.89         4

                 accuracy                           0.88        16
                macro avg       0.89      0.90      0.88        16
             weighted avg       0.91      0.88      0.88        16



In [22]:
print("=== REPORT DETALHADA ===")
print(classification_report(test_df["classe_detalhada"], detail_pred_test, zero_division=0))

=== REPORT DETALHADA ===
                                        precision    recall  f1-score   support

                 Agradecimento Simples       0.00      0.00      0.00         1
         Alteração de Plano / Contrato       0.00      0.00      0.00         0
                 Atualização Cadastral       0.00      0.00      0.00         1
          Criação / Gestão de Usuários       1.00      1.00      1.00         1
                Crítica de Usabilidade       0.00      0.00      0.00         0
            Customização de Relatórios       1.00      1.00      1.00         1
        Dúvidas sobre Fatura / Valores       1.00      1.00      1.00         1
         Erro de Autenticação e Acesso       0.00      0.00      0.00         0
           Erro de Sistema / Aplicação       0.00      0.00      0.00         2
               Falha de Banco de Dados       1.00      1.00      1.00         1
  Indisponibilidade / Queda do Serviço       0.00      0.00      0.00         0
              

# Salvando Modelos

In [25]:
joblib.dump(modelo_macro, "models\modelo_macro.joblib")
joblib.dump(modelos_especialistas, "models\modelos_especialistas.joblib")

['models\\modelos_especialistas.joblib']